# RSM8413 — Simple Data Cleaning & EDA (15 Predictors)

**Target:** `i9_health` — intention to self-isolate when feeling unwell (Yes / No / Not sure).  
**15 candidate predictors:** 8 context/demographic + 4 well-being + 3 COVID attitudes.  
**Scope:** Data cleaning, exploratory analysis, encoding and scaling only. NO kNN fitting or evaluation.

Run in VS Code with the source CSV in the **same folder**. The original responses are preserved for EDA. Preprocessing is fitted **only on the training set**.

**Why 15? / 为什么选 15？** This is an interpretable, manageable *starting set*, not a statistically optimal number. We include meaningful variables with populated data and avoid target proxies (`i11_health`), IDs (`RecordNo`) and mostly empty questions. Check predictive value later using cross-validation.


## 1. Imports & Data Overview / 导入与数据概况
**What:** Load the original CSV, standardize blank strings, view dimensions.  
**Output:** `df_raw` and basic inspection. **Next:** Assess missingness and select variables.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")
df_raw = pd.read_csv("COVID-19BehaviorData_CAN2022.csv", low_memory=False)
df_raw.info()
#identify missing values and replace them with NaN
df_raw = df_raw.replace(r"^\s*$", np.nan, regex=True)
print("Shape:", df_raw.shape)
print("Data types:\n", df_raw.dtypes)
df_raw.head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6430 entries, 0 to 6429
Columns: 512 entries, RecordNo to future_3
dtypes: float64(1), int64(3), object(508)
memory usage: 25.1+ MB
Shape: (6430, 512)
Data types:
 RecordNo         int64
endtime         object
qweek           object
i1_health      float64
i2_health       object
                ...   
future_1        object
future_2        object
had_covid_2     object
long_covid      object
future_3        object
Length: 512, dtype: object


,RecordNo,endtime,qweek,i1_health,i2_health,i7a_health,i3_health,i4_health,i5_health_1,i5_health_2,...,vac_man_96,vac_man_99,q_other,had_covid,vac_boost_beyond,future_1,future_2,had_covid_2,long_covid,future_3
0,41589,06/01/2022 18:02,week 44,NaN,0,NaN,NaN,NaN,NaN,NaN,...,No,No,__NA__,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,41731,03/01/2022 23:48,week 44,NaN,1,NaN,NaN,NaN,NaN,NaN,...,No,No,__NA__,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,41756,10/01/2022 04:53,week 44,NaN,0,NaN,NaN,NaN,NaN,NaN,...,Yes,No,"Oui bar , restaurent",NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,41846,01/01/2022 13:15,week 44,NaN,5,NaN,NaN,NaN,NaN,NaN,...,No,No,__NA__,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,41847,01/01/2022 19:12,week 44,NaN,150,NaN,NaN,NaN,NaN,NaN,...,No,No,__NA__,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,41867,01/01/2022 15:21,week 44,NaN,4,NaN,NaN,NaN,NaN,NaN,...,No,No,__NA__,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,41868,01/01/2022 20:02,week 44,NaN,14,NaN,NaN,NaN,NaN,NaN,...,No,No,__NA__,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,41920,02/01/2022 12:36,week 44,NaN,0,NaN,NaN,NaN,NaN,NaN,...,No,No,__NA__,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,41937,01/01/2022 00:11,week 44,NaN,0,NaN,NaN,NaN,NaN,NaN,...,No,No,__NA__,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,41951,01/01/2022 00:43,week 44,NaN,5,NaN,NaN,NaN,NaN,NaN,...,No,No,__NA__,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Missing Values / 缺失值
**What:** Check empty columns and missing-rate summary.  
**Output:** Data-based tables only. **Next:** Eliminate empty fields and distinguish non-response from numeric values.

In [2]:
missing_rate = (df_raw.isna().mean() * 100).sort_values(ascending=False)
print("Completely empty columns:", (missing_rate == 100).sum())
display(missing_rate.head(12).to_frame("Missing %"))

Completely empty columns: 353


,Missing %
ct8_6,100.0
ct5_2,100.0
ct7_2,100.0
ct7_1,100.0
ct6_other,100.0
ct6_99,100.0
ct6_96,100.0
ct6_6,100.0
ct6_5,100.0
ct6_4,100.0


## 3. Select 15 Predictors & Clean / 15 个变量的筛选与清洗

**Demographics / context (8):** age, gender, region, household_size, household_children, employment_status, cantril_ladder, qweek.  
**Well-being (4):** PHQ4_1–PHQ4_4 (four self-reported mood/anxiety items).  
**COVID attitudes (3):** vac2_1 (worry about COVID), vac2_2 (concern about vaccine side effects), vac7 (vaccine trust).

**What:** Keep interpretable columns, convert real numeric columns, and mark survey non-responses as missing.  
**Output:** `df_clean` with **16 columns: 15 predictors + 1 target**, suitable for EDA.  
**Next:** Use readable category labels for plots and crosstabs.

**Important:** These are correlational survey features; avoid interpreting them as causes. Survey items concerning mental well-being and vaccine opinions deserve appropriate caution in any public-health application.

In [ ]:
target = "i9_health"
num_cols = ["age", "cantril_ladder"]
cat_cols = ["gender", "region", "household_size", "household_children",
            "employment_status", "qweek", "PHQ4_1", "PHQ4_2",
            "PHQ4_3", "PHQ4_4", "vac2_1", "vac2_2", "vac7"]
features = num_cols + cat_cols  # 15 predictors

df_clean = df_raw[[target] + features].copy()
for col in num_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")
for col in cat_cols:
    df_clean[col] = df_clean[col].replace(
        ["Prefer not to say", "Don't know"], np.nan)
print("Predictors:", len(features), "Cleaned shape:", df_clean.shape)
display(df_clean.isna().sum().to_frame("Missing count"))
display(df_clean.head())

## 4. EDA: Target Distribution / 目标变量分布
**Data-based:** Counts and percentages. **Visual:** Target bar chart.  
**Purpose:** Identify class imbalance and treat Not sure separately.  
**Next:** Define No = positive class for a later binary model, without confusing intent with actual isolation behavior.

In [ ]:
display(df_clean[target].value_counts(dropna=False))
display((df_clean[target].value_counts(normalize=True) * 100).round(1))
df_clean[target].value_counts().reindex(["Yes", "No", "Not sure"]).plot(kind="bar")
plt.title("Self-isolation intention")
plt.show()

## 5. EDA: Categorical Predictors vs Target / 类别变量与目标
**Data-based:** Crosstabs and within-group percentages for all 13 categorical features.  
**Visual:** Two normalized stacked bars (employment, vaccine trust); optionally change the list to other features.  
**Purpose:** Compare **rates**, not only raw counts, and check subgroup sizes.  
**Next:** Prioritize useful predictors and report only clear descriptive findings.

In [ ]:
for col in cat_cols:
    tab = pd.crosstab(df_clean[col], df_clean[target])
    pct = pd.crosstab(df_clean[col], df_clean[target], normalize="index") * 100
    print("\n", col)
    display(tab)
    display(pct.round(1))

for col in ["employment_status", "vac7"]:
    tab = pd.crosstab(df_clean[col], df_clean[target], normalize="index")
    tab.reindex(columns=["Yes", "No", "Not sure"], fill_value=0).plot(
        kind="bar", stacked=True, figsize=(9, 4), title=f"Isolation intention by {col}")
    plt.ylabel("Proportion")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.show()

## 6. EDA: Numeric Distributions / 数值分布
**Data-based:** `describe()`. **Visual:** Histograms and boxplots of age and life satisfaction.  
**Purpose:** Identify scale, skew and unusual values.  
**Next:** Choose scaling; optionally investigate age groups.

In [ ]:
display(df_clean[num_cols].describe())
df_clean[num_cols].hist(bins=18, figsize=(9, 3))
plt.tight_layout()
plt.show()
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
for ax, col in zip(axes, num_cols):
    df_clean[[col]].boxplot(column=col, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

## 7. EDA: Numeric vs Target / 数值与目标
**Data-based:** Age-group target proportions. **Visual:** Age by target boxplot and age-group target proportions.  
**Purpose:** Compare distributions, not claim causation.  
**Next:** Use results for business interpretation; retain age as numeric for kNN.

In [ ]:
df_clean.boxplot(column="age", by=target)
plt.title("Age by isolation intention")
plt.show()

df_clean["age_group"] = pd.cut(df_clean["age"],
    bins=[0, 29, 44, 59, 120], labels=["18-29", "30-44", "45-59", "60+"])
age_pct = pd.crosstab(df_clean["age_group"], df_clean[target], normalize="index")
display((age_pct * 100).round(1))
age_pct.plot(kind="bar", stacked=True, title="Isolation intention by age group")
plt.ylabel("Proportion")
plt.xticks(rotation=0)
plt.show()

## 8. EDA: Multivariate Patterns / 多变量
**Data-based:** Average No rate and cell sizes by age group × employment.  
**Visual:** Heatmap. **Purpose:** Explore patterns obscured by looking at one variable alone.  
**Next:** Identify subgroups worth further investigation; small cells need caution.

In [ ]:
df_clean["not_isolate"] = df_clean[target].eq("No").astype(int)
heat = df_clean.pivot_table(index="age_group", columns="employment_status",
                            values="not_isolate", aggfunc="mean", observed=False)
counts = df_clean.pivot_table(index="age_group", columns="employment_status",
                              values="not_isolate", aggfunc="count", observed=False)
display(counts)
plt.figure(figsize=(10, 4))
plt.imshow(heat.to_numpy(dtype=float), aspect="auto", cmap="Blues", vmin=0, vmax=1)
plt.colorbar(label="No rate")
plt.xticks(range(len(heat.columns)), heat.columns, rotation=45, ha="right")
plt.yticks(range(len(heat.index)), heat.index)
plt.title("No rate by age group and employment (check group sizes)")
plt.tight_layout()
plt.show()

## 9. EDA: Correlation / 相关性
**Data-based:** Pearson correlation of actual numeric features. **Visual:** Small heatmap.  
**Purpose:** Look for redundancy; nominal categories should not be assigned arbitrary numeric codes to make correlations.  
**Next:** Consider feature redundancy when designing kNN distances.

In [ ]:
corr = df_clean[num_cols].corr()
display(corr)
plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar()
plt.xticks(range(len(corr.columns)), corr.columns)
plt.yticks(range(len(corr.index)), corr.index)
plt.title("Numeric feature correlations")
plt.show()

## 10. Split, Encode & Scale / 划分、编码、缩放
**What:** Keep only Yes/No for binary classification (No=1); stratified train/test split; one-hot encode categorical predictors and scale two numeric predictors **using training data only**.  
**Output:** `X_train`, `X_test`, `y_train`, `y_test`, `preprocessor`, `X_train_ready`, `X_test_ready`.  
**Next:** Give model lead original `X_train`/`X_test` and pipeline definition so encoding/scaling can be **refitted inside each CV fold**. Ready matrices are for initial sanity checks, not CV fitting.

Questionnaire response labels are categorical here. One-hot encoding avoids assuming that a response category's numerical code represents a valid Euclidean distance.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

model_df = df_clean[df_clean[target].isin(["Yes", "No"])].copy()
X = model_df[features]
y = model_df[target].map({"Yes": 0, "No": 1})
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())]), num_cols),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))]), cat_cols)
])

X_train_ready = preprocessor.fit_transform(X_train)
X_test_ready = preprocessor.transform(X_test)
print("Train rows:", X_train.shape[0], "Test rows:", X_test.shape[0])
print("Encoded features:", X_train_ready.shape[1])
print("Missing target:", y.isna().sum())

## Handoff / 给建模组员
- `df_clean`: 15 predictors plus original Yes/No/Not sure target, plus EDA-only columns `age_group` and `not_isolate`.
- `features`, `num_cols`, `cat_cols`: agreed predictor specifications.
- `X_train`, `X_test`, `y_train`, `y_test`: shared split (No = 1).
- `preprocessor`: must be nested inside an sklearn pipeline with kNN for grid search CV; do **not** cross-validate on `X_train_ready`.

**Important:** 15 is an initial interpretable set, not an optimum. PHQ-4 items are correlated and may double-count similar information in kNN; check this as part of feature selection. We deliberately exclude `i11_health` because it asks almost the same isolation-intention question as the target.